# Tiled Creative Upscaler → Modular Diffusers — E2E

Full validation: publish **PRIVATE** `remyxai/tiled-upscaler-flux-modular` → load via `trust_remote_code` →
**×2 and ×4 on a real photo** → three quantitative checks: (1) output resolution, (2) **added detail** via
Laplacian-variance sharpness vs the Lanczos-only control, (3) **no visible tile seams** (edge energy along
tile borders vs the image average, + a content-preservation LPIPS-free SSIM check against the resize).
Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 2 · Publish PRIVATE (upload block.py first)

In [ ]:
import os
from huggingface_hub import HfApi
for f in ["block.py", "modular_config.json", "modular_model_index.json"]:
    assert os.path.exists(f), f"upload {f} next to this notebook first."
api = HfApi(); REPO = "remyxai/tiled-upscaler-flux-modular"
api.create_repo(REPO, private=True, repo_type="model", exist_ok=True)
for f in ["block.py", "modular_config.json", "modular_model_index.json"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))

## 3 · Load + a real photo

In [ ]:
from diffusers import ModularPipeline
from PIL import Image
from io import BytesIO
from IPython.display import display
pipe = ModularPipeline.from_pretrained(REPO, trust_remote_code=True)
assert type(pipe.blocks).__name__ == "TiledUpscalerBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect TiledUpscalerBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

In [ ]:
import requests
IMG_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"  #@param {type:"string"}
try:
    src = Image.open(BytesIO(requests.get(IMG_URL, timeout=30).content)).convert("RGB")
except Exception as e:
    print("fetch failed, upload one:", e)
    from google.colab import files; up = files.upload(); src = Image.open(list(up.keys())[0]).convert("RGB")
src = src.resize((512, 512)); src.save("src.png")   # deliberately low-res input
print("source (low-res):", src.size); display(src)

## 4 · Milestone A — ×2 upscale

512² → 1024². With `tile_size=1024` that is a single tile, so this milestone checks the upscale + refine path;
the ×4 milestone below is what exercises the overlap blending (9 tiles).

In [ ]:
import torch
g = torch.Generator(DEV).manual_seed(0)
up2 = pipe(image="src.png", scale=2, tile_size=1024, tile_overlap=128, denoise_strength=0.4,
           num_inference_steps=28, guidance_scale=3.5, generator=g).images[0]
up2.save("up2.png")
print("[x2] source", src.size, "-> output", up2.size)
display(up2.resize((512, 512)))

## 5 · Milestone B — ×4 upscale (multi-tile + overlap blending)

512² → 2048². `tile_size=1024`, `tile_overlap=128` → a 3×3 grid with feathered overlaps: the case the seam
check below targets.

In [ ]:
g = torch.Generator(DEV).manual_seed(0)
up4 = pipe(image="src.png", scale=4, tile_size=1024, tile_overlap=128, denoise_strength=0.35,
           num_inference_steps=28, guidance_scale=3.5, generator=g).images[0]
up4.save("up4.png")
print("[x4] source", src.size, "-> output", up4.size)
display(up4.resize((512, 512)))

## 6 · Check 1 — output resolution

The output side must equal the input side × `scale` (both snapped down to a /16 multiple, per the VAE).

In [ ]:
def snap(v, q=16): return max(q, round(v) // q * q)
r1 = {"x2": (up2.size, (snap(512 * 2), snap(512 * 2))),
      "x4": (up4.size, (snap(512 * 4), snap(512 * 4)))}
for k, (got, want) in r1.items():
    print(f"  {k}: got {got}, want {want} -> {'PASS' if got == want else 'FAIL'}")
assert all(g == w for g, w in r1.values()), "resolution check failed"
print("resolution: PASS")

## 7 · Check 2 — added detail (Laplacian variance)


Sharpness = variance of the Laplacian, a standard detail/blur metric. The claim is that tiled refine adds
**real detail beyond a plain resize**, so the comparison target is the *Lanczos-only* resize of the same
source to the same size (not the low-res source, which trivially wins). Both are compared at the same
resolution; compare the ×2 and ×4 results against their own controls.

In [ ]:
import numpy as np
import torch.nn.functional as F
_LAP = torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]).view(1, 1, 3, 3)

def lap_var(img):
    """Variance of the Laplacian, measured on grayscale — a standard sharpness/detail metric."""
    x = torch.from_numpy(np.asarray(img.convert("L"), dtype=np.float32) / 255.0)[None, None]
    lap = F.conv2d(x, _LAP)
    return float(lap.var())

ctrl2 = src.resize((snap(512 * 2), snap(512 * 2)), Image.LANCZOS)   # Lanczos-only control
ctrl4 = src.resize((snap(512 * 4), snap(512 * 4)), Image.LANCZOS)
for name, out, ctrl in [("x2", up2, ctrl2), ("x4", up4, ctrl4)]:
    a, b = lap_var(out), lap_var(ctrl)
    print(f"  {name}: tiled={a:.2f}  lanczos-only={b:.2f}  ratio={a / max(b, 1e-9):.2f}x -> "
          f"{'PASS' if a > b else 'REVIEW'}")
print("added-detail:", "PASS" if lap_var(up2) > lap_var(ctrl2) and lap_var(up4) > lap_var(ctrl4) else "REVIEW")

## 8 · Check 3 — no visible tile seams

A seam is a discontinuity injected along a tile border, and tile borders sit **inside** the image — so the
right test is *relative*: mean gradient magnitude on the interior tile-border lines vs the image's own
average. A seam-free blend keeps the border lines unremarkable (ratio ≈ 1); a failed blend spikes it
(ratio ≫ 1). Computed on the **×4** output, where the 3×3 grid actually exercises the overlap.

In [ ]:
import numpy as np
import torch.nn.functional as F

def gray(img):
    return torch.from_numpy(np.asarray(img.convert("L"), dtype=np.float32) / 255.0)[None, None]

def border_ratio(img, scale, tile_size=1024, overlap=128):
    """Gradient energy on interior tile borders / image-average gradient energy."""
    g = gray(img)
    gy = F.conv2d(g, torch.tensor([[-1., 0., 1.]]).view(1, 1, 1, 3), padding=(0, 1))
    gx = F.conv2d(g, torch.tensor([[-1.], [0.], [1.]]).view(1, 1, 3, 1), padding=(1, 0))
    gm = (gx ** 2 + gy ** 2).sqrt()[0, 0]          # gradient magnitude per pixel
    H, W = gm.shape
    stride = tile_size - overlap
    ys = [y for y in range(stride, H - overlap, stride) if y < H]   # interior tile boundaries
    xs = [x for x in range(stride, W - overlap, stride) if x < W]
    lines, n = torch.zeros(()), 0
    for y in ys:
        lines = lines + gm[max(y - overlap // 2, 0):y + overlap // 2].mean(); n += 1
    for x in xs:
        lines = lines + gm[:, max(x - overlap // 2, 0):x + overlap // 2].mean(); n += 1
    return float(lines / max(n, 1) / gm.mean()), len(ys), len(xs)

r, ny, nx = border_ratio(up4, 4)
print(f"  x4: {ny} horizontal + {nx} vertical interior borders, seam ratio = {r:.2f} "
      f"(PASS < 1.5, REVIEW >= 1.5)")
print("seam-free blending:", "PASS" if r < 1.5 else "REVIEW")

## 9 · Check 4 — content preserved (didn't redraw the image)

Guard against *over*-denoising: the upscale must still be the same picture. SSIM (no extra deps) against the
Lanczos control at matching size — high = structure preserved.

In [ ]:
def ssim(img_a, img_b):
    a, b = gray(img_a), gray(img_b)
    mu_a, mu_b = F.avg_pool2d(a, 8), F.avg_pool2d(b, 8)
    s_a, s_b = F.avg_pool2d(a * a, 8) - mu_a ** 2, F.avg_pool2d(b * b, 8) - mu_b ** 2
    cov = F.avg_pool2d(a * b, 8) - mu_a * mu_b
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    m = ((2 * mu_a * mu_b + c1) * (2 * cov + c2)) / ((mu_a ** 2 + mu_b ** 2 + c1) * (s_a + s_b + c2))
    return float(m.mean())

for name, out, ctrl in [("x2", up2, ctrl2), ("x4", up4, ctrl4)]:
    s = ssim(out, ctrl)
    print(f"  {name}: SSIM vs Lanczos control = {s:.3f} (PASS > 0.55, REVIEW <= 0.55)")
print("content preservation:", "PASS" if ssim(up2, ctrl2) > 0.55 and ssim(up4, ctrl4) > 0.55 else "REVIEW")

## 10 · Result grid + the knob that matters

source · ×2 · ×4 side by side, then a `denoise_strength` sweep — that parameter is the whole trade-off: ↑ adds
detail but drifts from the source, ↓ stays faithful but restores less.

In [ ]:
from PIL import ImageDraw, ImageFont
panels = [("source", src), ("x2", up2), ("x4", up4)]
S = 512; W = len(panels) * S + (len(panels) + 1) * 10
row = Image.new("RGB", (W, S + 40), "white")
d = ImageDraw.Draw(row)
try: F_ = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 26)
except Exception: F_ = ImageFont.load_default()
for i, (name, im) in enumerate(panels):
    x = 10 + i * (S + 10); row.paste(im.resize((S, S)), (x, 0)); d.text((x + 8, S + 6), name, fill="black", font=F_)
row.save("tiled_upscaler_grid.png"); display(row)

In [ ]:
for st in [0.25, 0.4, 0.55]:
    g = torch.Generator(DEV).manual_seed(0)
    im = pipe(image="src.png", scale=2, tile_size=1024, tile_overlap=128, denoise_strength=st,
              num_inference_steps=28, guidance_scale=3.5, generator=g).images[0]
    print(f"  denoise_strength={st}: lap_var={lap_var(im):.2f}  SSIM={ssim(im, ctrl2):.3f}")

## Verdict

`loaded block: TiledUpscalerBlock` + **resolution PASS** + **Laplacian variance up vs the Lanczos control** +
**seam ratio < 1.5** on the multi-tile ×4 + SSIM > 0.55 = the tiled upscaler works as a Modular Diffusers
block. Optional cross-check: run `neuralwork/flux-tiled-upscaler` on the same source and eyeball parity.
Then flip the Hub repo public, add the Colab badge + the demo asset, and add it to the collection.